In [4]:
# Cell 1: ARAA-SEARCH VERSION (Drop-in replacement)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
import requests
from bs4 import BeautifulSoup
import re, time
import openai

# ARAA-SEARCH CONFIG (change only this)
ARAA_SEARCH_URL = "http://localhost:8080"  # Your Araa-Search URL
VLLM_URL = "http://127.0.0.1:5000/v1"
VLLM_API_KEY = "no-key"
VLLM_MODEL_NAME = "cpatonn/Qwen3-30B-A3B-Thinking-2507-AWQ-4bit"

# AUTO-DETECT VLLM MODEL
def get_vllm_model():
    from openai import OpenAI
    client = OpenAI(base_url=VLLM_URL, api_key=VLLM_API_KEY)
    models = client.models.list()
    return models.data[0].id if models.data else VLLM_MODEL_NAME

VLLM_MODEL = get_vllm_model()
print(f"✅ Araa-Search + VLLM({VLLM_MODEL})")

# Cell 2: ARAA-SEARCH TOOL (Adapted API)
@tool
def araa_search(query: str) -> str:
    """Search using Araa-Search engine"""
    try:
        # Araa-Search API (adjust endpoint if different)
        resp = requests.post(
            f"{ARAA_SEARCH_URL}/search",  # Common Araa endpoint
            json={"query": query, "num_results": 8},
            headers={"Content-Type": "application/json"},
            timeout=15
        )
        data = resp.json()
        
        # Handle common Araa response formats
        results = data.get("results") or data.get("data") or data.get("search_results", [])
        
        formatted = "ARAA-SEARCH RESULTS:\n\n"
        for i, result in enumerate(results[:6], 1):
            title = result.get("title") or result.get("name", "No title")
            url = result.get("url") or result.get("link", "")
            snippet = result.get("snippet") or result.get("description", "")[:180]
            
            formatted += f"{i}. **{title}**\n"
            formatted += f"   {url}\n"
            formatted += f"   {snippet}...\n\n"
        return formatted
        
    except Exception as e:
        # Fallback GET if POST fails
        try:
            resp = requests.get(f"{ARAA_SEARCH_URL}/search", params={"q": query}, timeout=12)
            data = resp.json()
            results = data.get("results", [])
            return "\n".join([f"{i}. {r.get('title')} ({r.get('url')})\n{r.get('snippet', '')[:150]}..."
                            for i, r in enumerate(results[:5])])
        except:
            return f"Araa-Search error: {str(e)}"

@tool
def get_content(urls: list[str]) -> str:
    """Fetch webpage content"""
    content = ""
    for url in urls[:3]:
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.content, "html.parser")
            for tag in soup(["script", "style"]): tag.decompose()
            text = re.sub(r'\s+', ' ', soup.get_text(strip=True))[:2200]
            title = soup.title.string or url.split("/")[-1]
            content += f"\n## {title}\n\n{text[:450]}...\n\n"
            time.sleep(0.4)
        except: continue
    return content or "No content retrieved"

@tool
def summarize_content(text: str, query: str) -> str:
    """VLLM summarization"""
    from openai import OpenAI
    client = OpenAI(base_url=VLLM_URL, api_key=VLLM_API_KEY)
    resp = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{"role": "user", "content": f"Query: '{query}'\n\nSummarize:\n\n{text[:3800]}" }],
        temperature=0.3, max_tokens=450
    )
    return resp.choices[0].message.content.strip()

tools = [araa_search, get_content, summarize_content]  # Changed tool name
print("✅ Araa-Search tools ready")

# Cell 3: v1.0.x ARAA AGENT

llm = ChatOpenAI(model=VLLM_MODEL, base_url=VLLM_URL, api_key=VLLM_API_KEY, temperature=0.7)
react_prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, react_prompt)

ARAA_AGENT = AgentExecutor(
    agent=agent, tools=tools, verbose=True,
    handle_parsing_errors=True, max_iterations=4
)

print("✅ Araa-Search agent ready!")

# Cell 4: USAGE (Identical)
def araa_research(query: str):
    """Araa-Search research"""
    return ARAA_AGENT.invoke({"input": query})

# Test
print("\n🧪 Araa-Search Test:")
result = araa_research("Latest vLLM developments")
print(result["output"])


✅ Araa-Search + VLLM(cpatonn/Qwen3-30B-A3B-Thinking-2507-AWQ-4bit)
✅ Araa-Search tools ready
✅ Araa-Search agent ready!

🧪 Araa-Search Test:


> Entering new AgentExecutor chain...
Okay, I need to find the latest developments in vLLM. Let me start by searching for recent information using the Araa-Search tool. The user mentioned vLLM, which is a library for efficient LLM serving. So, I'll use araa_search with the query "latest vLLM developments".

Wait, the tools available are araa_search, get_content, and summarize_content. First, I should search for the latest news or updates on vLLM. Let me run araa_search with "latest vLLM developments".

After getting the search results, I need to check the URLs from the search. Then, use get_content to fetch the content of those URLs. Once I have the content, I can use summarize_content to get a concise summary. But since the user is asking for the latest developments, maybe the search results will have the most recent info. Let me proceed step b

In [5]:
query="Explain the significance of Araa-Search integration with vLLM models."

In [8]:
# Verify Araa-Search
try:
    resp = requests.get(f"{ARAA_SEARCH_URL}/health", timeout=5)
    print("✅ Araa-Search healthy")
except:
    print("⚠️ Check Araa-Search URL/port")


⚠️ Check Araa-Search URL/port
